# BlackCode FR 260M - Entraînement dual-T4 Kaggle (6Md)

Pipeline optimisé max :
- Dataset 4 Go FineWeb2-HQ fra_Latn avec filtre français heuristique + qualité
- Tokenizer 24k français
- Train 6 Md tokens en dual-T4 DDP avec Muon + WSD + selective checkpoint + compile

Temps estimé : 1h30 build + 15h train 6Md = ~19h total (dans quota 30h)


In [ ]:
!pip install datasets lingua-language-detector tokenizers -q
!git clone https://github.com/Snowoo-2z/Llm-search.git /tmp/Llm-search || true
%cd /tmp/Llm-search
!ls configs/blackcode_fr_260m.json


In [ ]:
# Audit rapide 500 lignes
!python scripts/audit_fr_source.py --dataset epfml/FineWeb2-HQ:fra_Latn --limit 500 --sample-count 10


In [ ]:
# Build optimisé 4 Go
!MAX_MB=4000 MIN_QUALITY_SCORE=0.6 OUT=data/fr_260m HF_DATASET=epfml/FineWeb2-HQ:fra_Latn VOCAB=24576 SEQ_LEN=2048 bash scripts/build_fr_optimized.sh


In [ ]:
# Vérif pureté
!python scripts/eval_french_purity.py --corpus-dir data/fr_260m --limit 50
!cat data/fr_260m/meta.json | head -100


In [ ]:
# Train pilote 200 steps pour mesurer MFU
!python -m blackcode.train --config configs/blackcode_fr_260m.json --data data/fr_260m --out runs/fr_260m_pilot --batch-size 2 --grad-accum 8 --seq-len 2048 --max-steps 200 --optim adafactor --lr 3e-4 --checkpoint-policy selective --compile --eval-interval 100 --save-interval 100 --resume --log-flops


In [ ]:
# Train complet 6 Md en DDP dual-T4 (Kaggle)
# 6Md / (2*8*2048) = 183105 steps
!torchrun --nproc_per_node=2 -m blackcode.train --config configs/blackcode_fr_260m.json --data data/fr_260m --out runs/fr_260m_6b --batch-size 2 --grad-accum 8 --seq-len 2048 --max-steps 183105 --optim muon --muon-lr 0.02 --lr 6e-4 --schedule wsd --stable-frac 0.8 --checkpoint-policy selective --compile --eval-interval 1000 --save-interval 1000 --resume --log-flops --ddp
